In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("customers", defaultValue="customers", label="customers")
customers=dbutils.widgets.get("customers")

In [0]:
cust_schema=StructType([StructField("id", StringType(), True),
                        StructField("customer_id", StringType(), True),
                        StructField("full_name", StringType(), True),
                        StructField("email", StringType(), True),
                        StructField("phone", StringType(), True),
                        StructField("loyalty_tier",StringType(),True),
                        StructField("signup_date",DateType(),True),
                        StructField("created_at",StringType(),True),
                        StructField("updated_at",StringType(),True)])
overal_schema=StructType([StructField("data",ArrayType(cust_schema),True),StructField("ingestion_time",StringType(),True),StructField("batch_no",StringType(),True)])

In [0]:
df = (spark.readStream.format("cloudFiles")\
      .schema(overal_schema)\
            .option('cloudFiles.format', 'json')\
                  .option("cloudFiles.maxFilesPerTrigger",1)\
                        .option("multiLine", True)\
                              #.option("rescuedDataColumn", "_rescued_data")\
                                    .schema(overal_schema)\
                                          .load("abfss://raw@voltgridstorageaccount.dfs.core.windows.net/customers/"))

In [0]:
bronze_df=df.select(F.explode('data').alias('data'),F.col('ingestion_time'),F.col('batch_no'))\
    .select(F.col('data.*'),'ingestion_time','batch_no')

In [0]:
bronze_df=bronze_df.withColumn('created_at', F.col('created_at').try_cast('date'))\
    .withColumn('updated_at', F.col('updated_at').try_cast('date'))\
        .withColumn('ingestion_time',F.col('ingestion_time').try_cast('timestamp'))\
            .withColumn('id',F.col('id').cast('int'))\
                .withColumn('batch_no',F.col('batch_no').cast('int'))\
                    .withColumn('source_path',F.col('_metadata.file_path'))

In [0]:
(bronze_df.writeStream
    .format("completed")
    .outputMode("append")
    .option("checkpointLocation", "abfss://controls@voltgridstorageaccount.dfs.core.windows.net/checkpoints/customers/")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"voltgrid.bronze.{customers}")
    .awaitTermination())